# QSR Review Pipeline: 01 · Synthetic reviews (Bronze → Silver)

Generates **synthetic guest reviews** for a fictional QSR brand and lands them in the
same bronze and silver tables the rest of the pipeline expects. There is no external
API and no customer data in this pipeline.

| Layer | Table | Contents |
|-------|-------|----------|
| Bronze | `reviews_bronze` | Generated reviews, one row per review, no transforms |
| Silver | `reviews_silver` | Deduped, nulls handled, URLs and whitespace stripped |
| Dim | `store_locations` | Store id to City / State. `02_enrich` joins it for the "City, ST" store label |

Downstream, `02_enrich` reads `reviews_silver` and writes `reviews_gold`, and
`03_genie_agent` builds a Genie space over it.

> **Env:** set the `catalog` / `schema` widgets. Defaults are `jdub_demo.guest_sentiment`
> (the demo schema); point them at any catalog and schema you own.

In [ ]:
# --- Parameters (widgets) ---------------------------------------------------
dbutils.widgets.text("catalog", "jdub_demo", "Catalog")
dbutils.widgets.text("schema", "guest_sentiment", "Schema")
dbutils.widgets.text("n_reviews", "1200", "Reviews to generate")
dbutils.widgets.text("seed", "42", "Random seed (same seed, same reviews)")

CATALOG   = dbutils.widgets.get("catalog").strip()
SCHEMA    = dbutils.widgets.get("schema").strip()
N_REVIEWS = int(dbutils.widgets.get("n_reviews").strip())
SEED      = int(dbutils.widgets.get("seed").strip())
FQ = lambda t: f"{CATALOG}.{SCHEMA}.{t}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Target: {CATALOG}.{SCHEMA}")
print(f"  bronze -> {FQ('reviews_bronze')}")
print(f"  silver -> {FQ('reviews_silver')}")
print(f"  stores -> {FQ('store_locations')}")

## Bronze: generate synthetic reviews

Each review gets a target sentiment, one to three experience aspects (Speed, Cleanliness,
Order Accuracy, Quality, Service), and often a named product. Sentences come from
aspect-specific templates, so the AI step in `02_enrich` has real signal to extract.

A few messy cases are included on purpose (missing ratings, URLs, exact duplicate rows,
whitespace-only text, uneven spacing) so the silver step has something to clean. Three
stores run hotter on complaints so the dashboards have a visible spread.

In [ ]:
import random
import datetime as dt
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

rng = random.Random(SEED)

# Fictional store footprint. Location ids are synthetic.
STORES = [
    (1001, "Detroit", "MI"), (1002, "Detroit", "MI"), (1003, "Warren", "MI"),
    (1004, "Chicago", "IL"), (1005, "Chicago", "IL"), (1006, "Houston", "TX"),
    (1007, "Houston", "TX"), (1008, "Phoenix", "AZ"), (1009, "Las Vegas", "NV"),
    (1010, "Atlanta", "GA"), (1011, "Tampa", "FL"), (1012, "Indianapolis", "IN"),
    (1013, "Columbus", "OH"), (1014, "Charlotte", "NC"), (1015, "Dallas", "TX"),
    (1016, "Memphis", "TN"), (1017, "Cleveland", "OH"), (1018, "Cincinnati", "OH"),
    (1019, "Pittsburgh", "PA"), (1020, "San Antonio", "TX"),
]
PROBLEM_STORES = {1004, 1011, 1017}

ASPECTS  = ["Speed", "Cleanliness", "Order Accuracy", "Quality", "Service"]
PRODUCTS = ["Pizza", "Breadsticks", "Wings", "Cheese Bites", "Deep Dish", "Stuffed Crust",
            "Pepperoni", "Cheese", "Sauce", "Crust"]
ISSUE_PHRASES = ["soggy", "cold", "undercooked", "burnt", "greasy", "stale and dry", "bland"]
SIDES    = ["breadsticks", "dipping sauce", "wings", "cheese bites"]
TOPPINGS = ["onions", "olives", "pepperoni", "extra cheese"]

POSITIVE = {
    "Speed": ["Order was up in about {m} minutes, quick and easy.",
              "Drive-thru line moved fast today.",
              "Pickup was ready the moment I walked in."],
    "Cleanliness": ["Dining room was spotless and the tables were wiped down.",
                    "Restroom was clean and well stocked.",
                    "The store looked tidy even during the lunch rush."],
    "Order Accuracy": ["Everything in my order was exactly right.",
                       "They got my customizations perfect this time.",
                       "Order was complete, nothing missing."],
    "Quality": ["The {p} came out hot and cooked through.",
                "The {p} had great flavor and a fresh crust.",
                "The crust on the {p} was crisp and well browned."],
    "Service": ["Staff was friendly and checked on us.",
                "The cashier was patient and helpful.",
                "The manager came by to make sure everything was good."],
}
NEGATIVE = {
    "Speed": ["Waited {m} minutes for an order that should have taken 10.",
              "The drive-thru line barely moved and nobody said a word.",
              "Took forever to get my food at the counter."],
    "Cleanliness": ["Sticky tables and a dirty floor by the drink station.",
                    "The restroom was out of soap and looked neglected.",
                    "Trash bins were overflowing when I came in."],
    "Order Accuracy": ["They forgot the {side} and got the rest wrong.",
                       "Ordered a {p} with no {t} and it came with them anyway.",
                       "The {p} came with skimpy toppings and a small portion."],
    "Quality": ["The {p} was {issue} when it reached the table.",
                "The {p} tasted {issue}, not what I expected.",
                "Really disappointed with the {p}, it was {issue}."],
    "Service": ["The cashier was rude and never apologized for the wait.",
                "Nobody at the counter made eye contact.",
                "Felt ignored while I waited to place my order."],
}
NEUTRAL = ["Picked up an order for the family, nothing special either way.",
           "Food was about what I expected for a weeknight.",
           "Average visit, got what I ordered."]

START, END = dt.date(2024, 7, 1), dt.date(2026, 6, 22)


def _fill(template: str, product: str) -> str:
    return template.format(m=rng.randint(12, 25), p=product,
                           side=rng.choice(SIDES), t=rng.choice(TOPPINGS),
                           issue=rng.choice(ISSUE_PHRASES))


def make_review() -> tuple:
    loc_id, _, _ = rng.choices(STORES, weights=[3 if s[0] in PROBLEM_STORES else 1 for s in STORES])[0]
    p_neg = 0.5 if loc_id in PROBLEM_STORES else 0.25
    r = rng.random()
    if r < p_neg:
        sentiment = "Negative"
    elif r < p_neg + 0.40:
        sentiment = "Positive"
    elif r < p_neg + 0.52:
        sentiment = "Mixed"
    else:
        sentiment = "Neutral"

    if sentiment == "Neutral":
        sentences = [rng.choice(NEUTRAL)]
        rating = rng.choice([2, 3, 4])
    else:
        aspects = rng.sample(ASPECTS, k=rng.randint(1, 3))
        product = rng.choice(PRODUCTS)
        sentences = []
        for i, aspect in enumerate(aspects):
            if sentiment == "Positive":
                bank = POSITIVE
            elif sentiment == "Negative":
                bank = NEGATIVE
            else:  # Mixed: first aspect praised, the rest criticized
                bank = POSITIVE if i == 0 else NEGATIVE
            sentences.append(_fill(rng.choice(bank[aspect]), product))
        rating = {"Positive": rng.choice([4, 5, 5]), "Negative": rng.choice([1, 1, 2]),
                  "Mixed": 3}[sentiment]

    text = " ".join(sentences)
    if rng.random() < 0.05:
        text += f" Photos: https://example.com/photos/{rng.randint(10000, 99999)}"
    if rng.random() < 0.10:
        text = text.replace(". ", ".   ")
    if rng.random() < 0.01:
        text = "   "
    days = (END - START).days
    review_date = (START + dt.timedelta(days=rng.randint(0, days))).isoformat()
    rating_val = None if rng.random() < 0.04 else float(rating)
    return (review_date, rating_val, text, "SYNTHETIC", "REVIEW", None, loc_id)


BRONZE_SCHEMA = StructType([
    StructField("date", StringType()),
    StructField("rating", DoubleType()),
    StructField("review_text", StringType()),
    StructField("directoryType", StringType()),
    StructField("type", StringType()),
    StructField("directLink", StringType()),
    StructField("locationId", LongType()),
])

rows = [make_review() for _ in range(N_REVIEWS)]
rows += rng.sample(rows, k=max(1, N_REVIEWS // 50))  # exact duplicates for the dedup step
rng.shuffle(rows)
df = spark.createDataFrame(rows, BRONZE_SCHEMA)
print(f"Generated {df.count()} synthetic review rows across {len(STORES)} stores")
display(df.limit(10))

In [ ]:
# Land the generated reviews as bronze (overwrite the full snapshot each run).
(df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(FQ("reviews_bronze")))
print(f"Saved {df.count()} rows to {FQ('reviews_bronze')}")

## Silver: clean

Dedup (exact, then on `review_text` + `date` + `locationId`), handle nulls (flag missing
ratings, fill string nulls), strip URLs and normalize whitespace, and drop rows that are
empty after cleaning.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import re

df_s = spark.table(FQ("reviews_bronze"))
print(f"Bronze row count: {df_s.count()}")

# Dedup: exact, then on the natural key.
df_s = df_s.dropDuplicates()
df_s = df_s.dropDuplicates(["review_text", "date", "locationId"])
print(f"After dedup: {df_s.count()}")

# Drop empty review_text.
df_s = df_s.filter(F.col("review_text").isNotNull() & (F.trim(F.col("review_text")) != ""))

# Missing-rating flag + fill; string nulls -> 'unknown'.
df_s = (df_s
    .withColumn("rating_missing", F.col("rating").isNull())
    .withColumn("rating", F.coalesce(F.col("rating"), F.lit(0.0)))
    .withColumns({c: F.coalesce(F.col(c), F.lit("unknown")) for c in ["directoryType", "type"]}))

# Text clean: strip URLs, collapse whitespace.
@F.udf(StringType())
def clean_text(t):
    if t is None:
        return None
    t = re.sub(r"http\S+|www\.\S+", "", t)
    return re.sub(r"\s+", " ", t).strip()

df_s = df_s.withColumn("review_text_cleaned", clean_text(F.col("review_text")))
df_s = df_s.filter(F.col("review_text_cleaned").isNotNull() & (F.trim(F.col("review_text_cleaned")) != ""))

df_silver = df_s.select(
    F.col("date"), F.col("rating"), F.col("rating_missing"),
    F.col("review_text_cleaned").alias("review_text"),
    F.col("directoryType"), F.col("type"), F.col("locationId"))

(df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(FQ("reviews_silver")))
print(f"Saved {df_silver.count()} rows to {FQ('reviews_silver')}")
df_silver.printSchema()

## Store dimension

`02_enrich` joins this table to replace the raw location id with the "City, ST" label that
the Guest Sentiment app and the Genie space parse.

In [ ]:
stores_df = spark.createDataFrame(STORES, "location_id long, City string, State string")
(stores_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(FQ("store_locations")))
display(spark.table(FQ("store_locations")))